# 37. Do the new views earn a place in the stack?

**One variable against ledger row 59** (`stack_30_top9`, CV 0.967968): the member set. Same
logistic combiner, same `C=1.0`, same logit meta-features, same fold-wise protocol, same
folds. Row 59 is refit here as the reproduction check and as the paired baseline.

Five candidates, all produced 2026-08-21 and logged as rows 66 to 70:

| candidate | solo CV | vs its own baseline |
|---|---|---|
| `cat_native_c1` | 0.958943 | -0.007972 vs row 26 |
| `cat_native_c2` | 0.961358 | -0.005557 vs row 26 |
| `lgb_raw` | 0.963464 | -0.003318 vs row 17 |
| `xgb_raw` | 0.964218 | -0.002881 vs row 38 |
| `cat_raw` | 0.961420 | -0.005495 vs row 26 |

**Every one of them loses to its own baseline on all five folds.** That is not an argument
against membership and this repo has paid to learn it. The 2026-08-12 correction: a member's
own CV is close to irrelevant to whether a fitted combiner wants it. `neural` sits 0.0247
behind the worst GBDT here and carries +0.0912. Row 59's `pair_top9` was measured null as a
model and took the second largest coefficient in the stack.

## The pre-registered gate

Unchanged from rows 26 and 34, fixed before the run: **positive on at least 4 of 5 folds and
at least +0.00005 on the paired mean.** Membership is a different decision from the gate, and
rows 32 and 34 both kept sub-floor additions. Submission is a third decision. The three are
reported separately because bundling them into one condition was the error corrected in row 59.

## The prediction, written before the run

All five candidates are **representation changes**, which is the category the six-point rule in
`NOTES.md` scores highly and the category four consecutive null knob sweeps were not. The public
evidence is specific: on the identical fold split, the three no-encoder views take three of the
top four coefficients in a 177-model stack, above eight stronger boosted trees.

**I predict the five-candidate arm clears the floor**, and by more than `pair_top9`'s +0.000043,
because dropping the encoder changes the whole feature basis where pair encoding added nine
columns to it. I do **not** predict it closes the 0.0016 gap to the public field; our views are
0.0028 to 0.0070 behind their public counterparts and a combiner cannot invent signal its
members do not carry.

The honest case against: our views may be *too* weak. `cat_native_c1` at 0.958943 is 0.009
behind the stack itself, further behind than any member ever admitted here.

In [1]:
# 37_stack_views.ipynb
# Membership gate for the five vectors produced by notebooks 35 and 36.
# Runs locally: every member vector already lives in artifacts/oof.
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(b for b in [Path.cwd(), *Path.cwd().parents]
            if (b / "data" / "raw" / "train.csv").exists())
O = ROOT / "artifacts" / "oof"
S = ROOT / "submissions"

train = pd.read_csv(ROOT / "data" / "raw" / "train.csv")
test = pd.read_csv(ROOT / "data" / "raw" / "test.csv")
y = train["addicted_label"].to_numpy(np.int8)

# The fold vector is rebuilt rather than loaded, and then checked. A silently different
# fold vector is the one error here that produces a clean-looking wrong answer.
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=42).split(train, y)):
    folds[va] = i
assert (folds >= 0).all() and np.bincount(folds).sum() == len(train)
FOLD_SHA = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
assert FOLD_SHA == "ec282b0968059676", FOLD_SHA
print(f"train {len(train):,}  test {len(test):,}  folds {np.bincount(folds)}")
print(f"fold sha {FOLD_SHA}  VERIFIED")

train 691,369  test 296,302  folds [138274 138274 138274 138274 138273]
fold sha ec282b0968059676  VERIFIED


In [2]:
# Row 59's thirty, in row 59's order, then the five candidates.
BASE = [
    ("te42", "te_bag42"), ("te2024", "te_seed2024"), ("te7", "te_seed7"),
    ("te2025", "te_seed2025"), ("te13", "te_seed13"),
    ("anchor", "lgbm_default_anchor_seed42"), ("trees300", "lgbm_trees300_seed42"),
    ("trees1000", "lgbm_trees1000_seed42"), ("trees2000", "lgbm_trees2000_seed42"),
    ("lr010", "lgbm_lr01_n1000_seed42"), ("lr005", "lgbm_lr005_n2000_seed42"),
    ("lr003", "lgbm_lr003_n3333_seed42"),
    ("bag42", "lgbm_bag08_lr005_n2000_seed42"),
    ("bag2024", "lgbm_bag08_lr005_n2000_seed2024"),
    ("bag7", "lgbm_bag08_lr005_n2000_seed7"),
    ("bag2025", "lgbm_bag08_lr005_n2000_seed2025"),
    ("bag13", "lgbm_bag08_lr005_n2000_seed13"),
    ("neural", "neural"), ("cat42", "catboost_te"), ("cat2024", "catboost_te_seed2024"),
    ("cat7", "catboost_te_seed7"), ("cat2025", "catboost_te_seed2025"),
    ("cat13", "catboost_te_seed13"), ("neural_te", "neural_te"),
    ("xgb_te", "xgb_te"), ("xgb2024", "xgb_te_seed2024"), ("xgb7", "xgb_te_seed7"),
    ("xgb2025", "xgb_te_seed2025"), ("xgb13", "xgb_te_seed13"),
    ("pair_top9", "xgb_pair_top9"),
]
CAND = [("cat_nat_c1", "cat_native_c1"), ("cat_nat_c2", "cat_native_c2"),
        ("lgb_raw", "lgb_raw"), ("xgb_raw", "xgb_raw"), ("cat_raw", "cat_raw")]


def load(stem, kind):
    # OOF/test vector. Two naming conventions exist in artifacts/oof. The bare
    # "{stem}.npy" form is the OOF side only: the early LightGBM members never had a
    # test .npy written and their test side lives in submissions/. Falling back to the
    # bare name for kind="test" silently returns the OOF vector, which is caught by the
    # length assert below only because train and test differ in length.
    cands = [O / f"{stem}_{kind}.npy"]
    if kind == "oof":
        cands.append(O / f"{stem}.npy")
    for c in cands:
        if c.exists():
            return np.load(c)
    if kind == "test" and (S / f"{stem}.csv").exists():
        df = pd.read_csv(S / f"{stem}.csv")
        # A csv written in a different row order blends perfectly cleanly and is
        # undetectable in the score. Checked rather than assumed.
        assert (df["id"].to_numpy() == test["id"].to_numpy()).all(), f"id order {stem}"
        return df["addicted_label"].to_numpy()
    raise FileNotFoundError(f"{stem} {kind}")


MEM = BASE + CAND
names = [n for n, _ in MEM]
Poof = {n: load(s, "oof") for n, s in MEM}
Ptest = {n: load(s, "test") for n, s in MEM}

for n in names:
    assert Poof[n].shape == (len(train),), n
    assert Ptest[n].shape == (len(test),), n
    assert np.isfinite(Poof[n]).all() and np.isfinite(Ptest[n]).all(), n
    # A partially failed run leaves a constant fold, which blends silently.
    assert min(np.ptp(Poof[n][folds == f]) for f in range(5)) > 0, f"dead fold in {n}"

# Exact-duplicate quarantine. A duplicated array silently DOUBLES that model's weight.
# Row 59 found xgb_pair_base bit-identical to xgb_te this way and excluded it.
seen = {}
for n in names:
    h = hashlib.md5(np.ascontiguousarray(Poof[n]).tobytes()).hexdigest()
    assert h not in seen, f"{n} is bit-identical to {seen[h]}"
    seen[h] = n
print(f"{len(names)} vectors loaded, no exact duplicates")

35 vectors loaded, no exact duplicates


In [3]:
def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-9, 1 - 1e-9)
    return np.clip(np.log(p / (1 - p)), -30, 30)


Loof = np.column_stack([logit(Poof[n]) for n in names])
Ltest = np.column_stack([logit(Ptest[n]) for n in names])
IDX = {n: i for i, n in enumerate(names)}
BASE30 = [IDX[n] for n, _ in BASE]

print("candidate solo CV, and disagreement with the members it most resembles:")
print(f"  {'candidate':12} {'solo CV':>10} {'rho vs xgb_te':>15} {'rho vs cat42':>14}")
for n, _ in CAND:
    cv = np.mean([roc_auc_score(y[folds == f], Poof[n][folds == f]) for f in range(5)])
    r1 = pd.Series(Poof[n]).corr(pd.Series(Poof["xgb_te"]), method="spearman")
    r2 = pd.Series(Poof[n]).corr(pd.Series(Poof["cat42"]), method="spearman")
    print(f"  {n:12} {cv:10.6f} {r1:15.6f} {r2:14.6f}")

# For scale: how decorrelated are two members that everyone agrees are near-copies?
r_seed = pd.Series(Poof["xgb_te"]).corr(pd.Series(Poof["xgb2024"]), method="spearman")
print(f"\n  for scale, xgb_te vs xgb_te_seed2024 (same model, different seed): {r_seed:.6f}")
print("  NOTES.md refuted reasoning from Spearman to blend value on 66 pairs at r=+0.143.")
print("  This table is context, not a prediction.")

candidate solo CV, and disagreement with the members it most resembles:
  candidate       solo CV   rho vs xgb_te   rho vs cat42


  cat_nat_c1     0.958943        0.969857       0.979956


  cat_nat_c2     0.961358        0.976628       0.985469


  lgb_raw        0.963464        0.983092       0.977006


  xgb_raw        0.964218        0.986133       0.979289


  cat_raw        0.961420        0.979227       0.983819



  for scale, xgb_te vs xgb_te_seed2024 (same model, different seed): 0.997344
  NOTES.md refuted reasoning from Spearman to blend value on 66 pairs at r=+0.143.
  This table is context, not a prediction.


In [4]:
def run(cols):
    # Fold-wise logistic combiner. No weight is ever fitted on a row it is scored on.
    oof = np.zeros(len(train))
    tst = np.zeros((5, len(test)))
    cf = np.zeros((5, len(cols)))
    nit = []
    for f in range(5):
        tr, va = folds != f, folds == f
        clf = LogisticRegression(C=1.0, max_iter=2000).fit(Loof[np.ix_(tr, cols)], y[tr])
        # A non-converged lbfgs fit reads HIGHER than the truth, so convergence is
        # asserted rather than hoped for. Added 2026-08-21 after the public review
        # flagged it; measured at 38 to 40 iterations, so it has never been close.
        nit.append(int(np.max(clf.n_iter_)))
        oof[va] = clf.decision_function(Loof[np.ix_(va, cols)])
        tst[f] = clf.decision_function(Ltest[:, cols])
        cf[f] = clf.coef_[0]
    assert max(nit) < 2000, f"combiner did not converge, {nit}"
    per = np.array([roc_auc_score(y[folds == f], oof[folds == f]) for f in range(5)])
    return per, tst, cf, max(nit)


ARMS = {"30_row59": BASE30}
for n, _ in CAND:
    ARMS[f"31_{n}"] = BASE30 + [IDX[n]]
ARMS["35_all5"] = BASE30 + [IDX[n] for n, _ in CAND]

res = {a: run(cols) for a, cols in ARMS.items()}
per = {a: r[0] for a, r in res.items()}

ROW59_CV = 0.967968
repro = per["30_row59"].mean() - ROW59_CV
print(f"reproduction of row 59: {per['30_row59'].mean():.6f} vs {ROW59_CV:.6f}"
      f"  delta {repro:+.2e}   {'REPRODUCED' if abs(repro) < 1e-4 else 'FAILED'}")
print(f"combiner max n_iter across all arms: {max(r[3] for r in res.values())} of 2000\n")

hdr = " ".join(f"{'fold ' + str(i):>9}" for i in range(5))
print(f"{'arm':14} {hdr} {'mean':>10} {'sd':>9}")
for a in ARMS:
    print(f"{a:14} " + " ".join(f"{v:9.6f}" for v in per[a])
          + f" {per[a].mean():10.6f} {per[a].std():9.6f}")

reproduction of row 59: 0.967968 vs 0.967968  delta +1.49e-07   REPRODUCED
combiner max n_iter across all arms: 45 of 2000

arm               fold 0    fold 1    fold 2    fold 3    fold 4       mean        sd
30_row59        0.967321  0.968092  0.968274  0.968524  0.967630   0.967968  0.000436
31_cat_nat_c1   0.967348  0.968119  0.968272  0.968531  0.967666   0.967987  0.000425
31_cat_nat_c2   0.967319  0.968090  0.968279  0.968531  0.967630   0.967970  0.000439
31_lgb_raw      0.967318  0.968092  0.968273  0.968522  0.967629   0.967967  0.000436
31_xgb_raw      0.967350  0.968113  0.968280  0.968547  0.967663   0.967990  0.000430
31_cat_raw      0.967321  0.968106  0.968288  0.968532  0.967627   0.967975  0.000441
35_all5         0.967460  0.968232  0.968365  0.968688  0.967803   0.968110  0.000432


In [5]:
FLOOR_MEAN, FLOOR_FOLDS = 0.00005, 4
base_per = per["30_row59"]

print("Paired against row 59's thirty. The gate is >= +0.00005 mean AND >= 4/5 folds.\n")
print(f"{'arm':14} {'paired mean':>13} {'paired sd':>11} {'folds':>7} {'t(4)':>8}  gate")
gate = {}
for a in ARMS:
    if a == "30_row59":
        continue
    d = per[a] - base_per
    wins = int((d > 0).sum())
    sd = d.std(ddof=1)
    t = d.mean() / (sd / np.sqrt(5)) if sd > 0 else float("inf")
    fired = bool(d.mean() >= FLOOR_MEAN and wins >= FLOOR_FOLDS)
    gate[a] = fired
    print(f"{a:14} {d.mean():+13.6f} {sd:11.6f} {wins:5d}/5 {t:8.2f}"
          f"  {'FIRES' if fired else 'under floor'}")

Paired against row 59's thirty. The gate is >= +0.00005 mean AND >= 4/5 folds.

arm              paired mean   paired sd   folds     t(4)  gate
31_cat_nat_c1      +0.000019    0.000016     4/5     2.66  under floor
31_cat_nat_c2      +0.000002    0.000004     3/5     0.92  under floor
31_lgb_raw         -0.000001    0.000001     0/5    -2.86  under floor
31_xgb_raw         +0.000022    0.000010     5/5     4.80  under floor
31_cat_raw         +0.000007    0.000008     4/5     1.88  under floor
35_all5            +0.000141    0.000032     5/5     9.92  FIRES


In [6]:
# Coefficients of the best arm, which is where the result actually lives. Row 59's
# lesson: a model that is null on its own can still take a large weight, and where that
# weight comes FROM is the thing worth reading.
best = max((a for a in ARMS if a != "30_row59"), key=lambda a: per[a].mean())
print(f"best arm by CV: {best}   {per[best].mean():.6f}\n")

cols_b, cols_0 = ARMS[best], ARMS["30_row59"]
cb = res[best][2].mean(axis=0)
c0 = res["30_row59"][2].mean(axis=0)
base_map = {names[c]: c0[i] for i, c in enumerate(cols_0)}

rows = []
for i, c in enumerate(cols_b):
    n = names[c]
    was = base_map.get(n, float("nan"))
    rows.append({"member": n, "coef": cb[i], "was": was, "shift": cb[i] - was})
tab = pd.DataFrame(rows).sort_values("coef", ascending=False)
print(tab.to_string(index=False, float_format=lambda v: f"{v:+.4f}"))

new = set(n for n, _ in CAND) & set(tab.member)
gained = tab[tab.member.isin(new)]["coef"].sum()
lost = -tab[~tab.member.isin(new)]["shift"].sum()
print(f"\nnew members carry {gained:+.4f} in total")
print(f"the existing thirty give up {lost:+.4f} of weight between them")
if abs(gained) > 1e-12:
    print(f"substitution covers {100 * lost / gained:.0f} percent of the new weight")

best arm by CV: 35_all5   0.968110

    member    coef     was   shift
cat_nat_c2 +0.2939     NaN     NaN
 pair_top9 +0.1921 +0.2061 -0.0141
   xgb_raw +0.1661     NaN     NaN
 neural_te +0.1219 +0.1226 -0.0008
     lr003 +0.0905 +0.1051 -0.0145
   xgb2024 +0.0761 +0.0898 -0.0137
     lr005 +0.0750 +0.0860 -0.0110
    neural +0.0732 +0.0722 +0.0010
   cat2024 +0.0728 +0.0600 +0.0128
     cat13 +0.0617 +0.0449 +0.0169
   cat2025 +0.0575 +0.0421 +0.0154
      xgb7 +0.0567 +0.0720 -0.0153
     cat42 +0.0564 +0.0381 +0.0184
     xgb13 +0.0563 +0.0690 -0.0127
      bag7 +0.0482 +0.0669 -0.0188
     bag13 +0.0457 +0.0662 -0.0206
      cat7 +0.0438 +0.0280 +0.0158
   xgb2025 +0.0368 +0.0506 -0.0139
   bag2025 +0.0281 +0.0423 -0.0142
    xgb_te +0.0269 +0.0415 -0.0146
     bag42 +0.0228 +0.0663 -0.0435
   lgb_raw +0.0136     NaN     NaN
   bag2024 +0.0087 +0.0149 -0.0062
    te2025 +0.0062 +0.0045 +0.0016
 trees1000 +0.0051 +0.0057 -0.0006
      te13 +0.0044 +0.0043 +0.0001
    te2024 +0.0004 

In [7]:
# Three decisions, kept separate. Bundling them was the error corrected in row 59.
print("1. GATE")
for a, f in gate.items():
    print(f"     {a:14} {'FIRES' if f else 'under floor'}")

print("\n2. MEMBERSHIP")
print("   A sub-floor addition is still kept and logged as negligible: row 32 kept four")
print("   CatBoost seeds at +0.000014 and row 34 kept neural_te at +0.000043. Membership")
print("   follows the sign and the fold count, not the floor.")
keep = [a for a in ARMS if a != "30_row59"
        and (per[a] - base_per).mean() > 0
        and int(((per[a] - base_per) > 0).sum()) >= 4]
print(f"   arms positive and >= 4/5 folds: {keep if keep else 'none'}")
print(f"   carried forward: {best} at {per[best].mean():.6f}")

print("\n3. SUBMISSION")
SUB = S / "stack_views_35.csv"
if per[best].mean() > ROW59_CV:
    p = res[best][1].mean(axis=0)
    sub = pd.DataFrame({"id": test["id"].to_numpy(),
                        "addicted_label": (np.argsort(np.argsort(p)) + 0.5) / len(p)})
    assert len(sub) == len(test) and np.isfinite(sub["addicted_label"]).all()
    sub.to_csv(SUB, index=False)
    print(f"   wrote {SUB.name}, {len(sub):,} rows,"
          f" {sub['addicted_label'].nunique():,} distinct")
    print("   AUC reads order only, so the rank transform changes nothing and keeps the")
    print("   file comparable with the earlier stack submissions.")
else:
    print(f"   no submission: best arm {per[best].mean():.6f} does not beat row 59")

print(f"\nledger lines:\n  name    stack_{best}\n  cv_mean {per[best].mean():.6f}"
      f"\n  cv_std  {per[best].std():.6f}")

1. GATE
     31_cat_nat_c1  under floor
     31_cat_nat_c2  under floor
     31_lgb_raw     under floor
     31_xgb_raw     under floor
     31_cat_raw     under floor
     35_all5        FIRES

2. MEMBERSHIP
   A sub-floor addition is still kept and logged as negligible: row 32 kept four
   CatBoost seeds at +0.000014 and row 34 kept neural_te at +0.000043. Membership
   follows the sign and the fold count, not the floor.
   arms positive and >= 4/5 folds: ['31_cat_nat_c1', '31_xgb_raw', '31_cat_raw', '35_all5']
   carried forward: 35_all5 at 0.968110

3. SUBMISSION


   wrote stack_views_35.csv, 296,302 rows, 296,302 distinct
   AUC reads order only, so the rank transform changes nothing and keeps the
   file comparable with the earlier stack submissions.

ledger lines:
  name    stack_35_all5
  cv_mean 0.968110
  cv_std  0.000432
